# 1. Подготовленные корпуса

Четыре синтетических корпуса с известным числом тем K и семь реальных корпусов на шести языках (раздел 5 статьи,
таблица 4). Каждый корпус лежит в `data/<корпус>/`:

| файл | содержание |
|---|---|
| `bow.npz` | мешок слов, разреженная матрица документы × слова (scipy CSR) |
| `vocab.txt` | словарь, по слову в строке, в порядке столбцов `bow.npz` |
| `stats.json` | число документов, размер словаря, число токенов, число рубрик |
| `docs.txt.gz` | те же документы токенами через пробел — вход сэмплера Гиббса для LDA |
| `w2v_vectors.npy` | векторы word2vec словаря, по которым считается силуэт (раздел 4.5) |
| `labels.npy` | рубрики документов (где они есть) |
| `phi_true.npy`, `truth.csv`, `popularity.npy` | синтетика: истинные темы, темы и доли каждого документа, популярность тем |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import json
import numpy as np
import pandas as pd
from rtm.data import CORPORA, DATA, load_bow, stats

rows = []
for c, (title, K, lang) in CORPORA.items():
    s = stats(c)
    rows.append({"корпус": c, "название": title, "язык": lang, "документов": s["docs"], "словарь V": s["V"],
                 "токенов": s["tokens"], "K / рубрик": K})
pd.DataFrame(rows)

,корпус,название,язык,документов,словарь V,токенов,K / рубрик
0,synth_K6,"Synthetic, 6 equal topics",—,6000,11229,436258,6
1,synth_k,"Synthetic, 12 equal topics",—,6000,11229,435672,12
2,synth_K20,"Synthetic, 20 equal topics",—,6000,11229,431789,20
3,synth_K12_uneq,"Synthetic, 12 unequal topics",—,6000,11229,435286,12
4,lenta_E2,Lenta,ru,8650,11229,627347,10
5,news20_clean,20 Newsgroups,en,15720,15347,1273183,20
6,livedoor,Livedoor,ja,7365,17101,1392440,9
7,thucnews,THUCNews,zh,10460,30000,2656934,14
8,gnad10,GNAD10,de,10252,27663,1830543,9
9,sanad,SANAD,ar,10470,9830,2070391,7


## Синтетические корпуса

Истинные темы — темы LDA (сэмплирование Гиббса, α = β = 0.01, 200 итераций), обученной на Lenta при T = K. Документ
содержит от одной до четырёх тем с долями θ ~ Dirichlet(2) и минимальной долей 0.1; длины документов взяты из Lenta;
6000 документов. Ниже корпуса порождаются заново из `phi_true.npy` и сравниваются с поставляемыми.

In [3]:
from rtm.synth import make_corpus

X0, _ = load_bow(DATA / "lenta_E2")
lengths = np.asarray(X0.sum(1)).ravel().astype(int)
for c in ("synth_K6", "synth_k", "synth_K20", "synth_K12_uneq"):
    P = np.load(DATA / c / "phi_true.npy")
    X, truth = make_corpus(P, lengths, uneven=stats(c)["uneven"], legacy_synth_k=(c == "synth_k"))
    Y, _ = load_bow(DATA / c)
    print(f"{c}: K = {P.shape[0]}, расхождений с поставляемым корпусом: {(X != Y).nnz}")
truth.head()

synth_K6: K = 6, расхождений с поставляемым корпусом: 0


synth_k: K = 12, расхождений с поставляемым корпусом: 0


synth_K20: K = 20, расхождений с поставляемым корпусом: 0


synth_K12_uneq: K = 12, расхождений с поставляемым корпусом: 0


,k_true,topics,theta
0,4,9 6 0 7,0.3693 0.2504 0.2076 0.1727
1,1,1,1.0000
2,1,0,1.0000
3,2,2 1,0.5221 0.4779
4,2,6 0,0.5747 0.4253


## Векторы слов для силуэта

`w2v_vectors.npy` — векторы skip-gram word2vec (размерность 100, окно 20, 10 эпох), обученного на перемешанных мешках
слов самого корпуса, нормированные к единичной длине. Многопоточное обучение word2vec не воспроизводится до бита,
поэтому для повторения таблиц статьи используются сохранённые векторы; `rtm.criteria.train_word_vectors` обучает
их заново.

In [4]:
from rtm.data import word_vectors

E = word_vectors("lenta_E2")
X, vocab = load_bow(DATA / "lenta_E2")
w = vocab.index("футбол")
nearest = np.argsort(-(E @ E[w]))[:8]
print(vocab[w], "->", [vocab[i] for i in nearest[1:]])

футбол -> ['футбольный', 'чемпионат', 'уефа', 'полузащитник', 'фифа', 'матч', 'турнир']
